In [1]:
!pip install yfinance openpyxl plotly --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Módulo 1: Datos y Optimización de Markowitz

Este notebook realiza la descarga de datos históricos de un conjunto de activos, calcula sus retornos diarios logarítmicos, estima sus retornos medios anualizados ($\mu$) y la matriz de covarianza ($\Sigma$), y finalmente realiza la optimización de portafolios de Markowitz (SLSQP) para maximizar el Sharpe Ratio. También calcula la Frontera Eficiente y simula la evolución de la riqueza histórica bajo tres estrategias diferentes de rebalanceo y costos de transacción.

In [2]:
import os
import json
import io
import time
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import minimize

# Crear carpeta de salidas si no existe
os.makedirs("salidas", exist_ok=True)

# Parámetros globales
TICKERS = ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']
FECHA_INICIO = '2015-01-01'
FECHA_FIN = '2024-12-31'
CAPITAL_INICIAL = 100000
RF = 0.0
DIAS_ANIO = 252

## 1. Descarga y Limpieza de Datos

Descargamos los precios históricos de cierre (`Close`) de los activos seleccionados desde Yahoo Finance y rellenamos cualquier valor faltante usando un método forward fill y backward fill.

In [3]:
print(f"Descargando datos para tickers: {TICKERS}")
descarga = yf.download(TICKERS, start=FECHA_INICIO, end=FECHA_FIN, progress=False)

# Asegurar procesamiento correcto del DataFrame
if isinstance(descarga, pd.DataFrame) and not descarga.empty:
    if 'Close' in descarga.columns:
        precios = descarga['Close']
    else:
        precios = descarga
else:
    raise ValueError("Error al descargar los precios históricos.")

# Limpieza
precios = precios.ffill().bfill()
TICKERS_VALIDOS = list(precios.columns)
N = len(TICKERS_VALIDOS)

print(f"Datos descargados con éxito. Activos válidos: {TICKERS_VALIDOS}")
precios.head()

Descargando datos para tickers: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']


Datos descargados con éxito. Activos válidos: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']


Ticker,ABX.TO,BHP,BVN,FSM,VOLCABC1.LM
Date,,,,,
2015-01-02,10.361857,20.644577,8.838105,4.68,0.663575
2015-01-05,10.321572,19.910683,9.368210,4.91,0.654358
2015-01-06,10.724442,19.784750,9.733798,5.09,0.654358
2015-01-07,10.595519,19.971476,9.624121,4.98,0.645142
2015-01-08,10.305452,20.271116,9.395628,4.89,0.645142


## 2. Cálculo de Retornos, Retorno Esperado Anualizado (Mu) y Riesgo (Sigma)

Calculamos los retornos diarios logarítmicos y los anualizamos multiplicando por el número de días de negociación del año (252).

In [4]:
retornos = pd.DataFrame(np.log(precios / precios.shift(1))).dropna()
mu = retornos.mean().to_numpy(dtype=float) * DIAS_ANIO
Sigma = retornos.cov().to_numpy(dtype=float) * DIAS_ANIO

print("Vector de retornos anualizados (Mu):")
for t, m in zip(TICKERS_VALIDOS, mu):
    print(f"  {t}: {m*100:.2f}%")

print("\nMatriz de Covarianza Anualizada (Sigma):")
print(Sigma)

Vector de retornos anualizados (Mu):
  ABX.TO: 7.17%
  BHP: 8.08%
  BVN: 2.48%
  FSM: -1.04%
  VOLCABC1.LM: -11.47%

Matriz de Covarianza Anualizada (Sigma):
[[0.14482636 0.02746382 0.10163427 0.14038463 0.02553678]
 [0.02746382 0.11322594 0.0640738  0.06967429 0.04511363]
 [0.10163427 0.0640738  0.23521711 0.17006877 0.06734871]
 [0.14038463 0.06967429 0.17006877 0.35473091 0.05182665]
 [0.02553678 0.04511363 0.06734871 0.05182665 0.28676261]]


## 3. Algoritmo de Optimización de Markowitz (SLSQP)

Definimos las funciones de cálculo del portafolio (retorno, riesgo, Sharpe Ratio) y configuramos la optimización con restricciones usando SLSQP.

In [5]:
def estadisticas(w):
    w = np.asarray(w, dtype=float)
    r = float(w @ mu)
    v = float(np.sqrt(w @ Sigma @ w))
    sh = (r - RF) / v if v > 0 else -1e9
    return r, v, sh

def neg_sharpe(w):
    return -estadisticas(w)[2]

# Restricciones y límites
restriccion_suma = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
limites = tuple((0.0, 1.0) for _ in range(N))
w0 = np.ones(N) / N

# Optimización para encontrar el Máximo Sharpe
res = minimize(neg_sharpe, w0, method='SLSQP', bounds=limites, constraints=restriccion_suma)
w_sharpe = res.x
r_s, v_s, sh_s = estadisticas(w_sharpe)

print("--- PORTAFOLIO ÓPTIMO (MÁXIMO SHARPE) ---")
print(f"Sharpe Ratio: {sh_s:.4f}")
print(f"Retorno Esperado Anualizado: {r_s*100:.2f}%")
print(f"Volatilidad Anualizada (Riesgo): {v_s*100:.2f}%")
print("Pesos asignados:")
for t, w in zip(TICKERS_VALIDOS, w_sharpe):
    print(f"  {t}: {w*100:.2f}%")

# Calcular frontera eficiente
def calcular_frontera(_mu, _Sigma, _w0, _limites, _N):
    frontera = []
    objetivos = np.linspace(_mu.min() + 0.001, _mu.max() - 0.001, 100)
    for obj in objetivos:
        cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},
                {'type': 'eq', 'fun': lambda w: w @ _mu - obj})
        res_f = minimize(lambda w: float(w @ _Sigma @ w), _w0, method='SLSQP', bounds=_limites, constraints=cons)
        r_f, v_f, sh_f = estadisticas(res_f.x)
        frontera.append((v_f, r_f))
    return np.array(frontera)

frontera = calcular_frontera(mu, Sigma, w0, limites, N)
vol_ind = np.sqrt(np.diag(Sigma))

--- PORTAFOLIO ÓPTIMO (MÁXIMO SHARPE) ---
Sharpe Ratio: 0.2780
Retorno Esperado Anualizado: 7.74%
Volatilidad Anualizada (Riesgo): 27.83%
Pesos asignados:
  ABX.TO: 37.73%
  BHP: 62.27%
  BVN: 0.00%
  FSM: 0.00%
  VOLCABC1.LM: 0.00%


## 4. Visualización de la Frontera Eficiente

Graficamos la frontera eficiente interactiva de Markowitz junto con los activos individuales y resaltamos el portafolio de Máximo Sharpe.

In [6]:
fig_final = go.Figure()
fig_final.add_trace(go.Scatter(x=frontera[:, 0], y=frontera[:, 1], mode='lines', name='Frontera Eficiente', line=dict(color='blue')))
fig_final.add_trace(go.Scatter(x=vol_ind, y=mu, mode='markers+text', name='Acciones', text=TICKERS_VALIDOS, textposition='top center', marker=dict(color='gray', size=10)))
fig_final.add_trace(go.Scatter(x=[v_s], y=[r_s], mode='markers', name=f'Máx Sharpe ({sh_s:.2f})', marker=dict(color='red', symbol='star', size=15)))
fig_final.update_layout(
    title='Frontera Eficiente de Markowitz y Portafolio Óptimo',
    xaxis_title='Riesgo (Volatilidad Anualizada)',
    yaxis_title='Retorno Anualizado',
    template='plotly_dark'
)
fig_final.show()

## 5. Análisis de Composición y Riesgo del Portafolio Óptimo

Generamos gráficos interactivos adicionales para la composición del portafolio (Dona), desempeño individual de los activos (Barras) y la matriz de correlación (Heatmap).

In [7]:
# 1. Gráfico de Dona de Pesos
df_pesos = pd.DataFrame({'Ticker': TICKERS_VALIDOS, 'Peso (%)': (w_sharpe * 100).round(2)})
df_pesos_filtrado = df_pesos[df_pesos['Peso (%)'] > 0.01]
fig_dona = px.pie(
    df_pesos_filtrado, 
    values='Peso (%)', 
    names='Ticker', 
    title='Composición del Portafolio Óptimo (Pesos %)', 
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_dona.update_layout(template='plotly_dark')
fig_dona.show()

# 2. Desempeño Individual de los Activos
df_activos = pd.DataFrame({
    'Ticker': TICKERS_VALIDOS,
    'Retorno Anualizado (%)': (mu * 100).round(2),
    'Volatilidad Anualizada (%)': (vol_ind * 100).round(2)
})
df_long = df_activos.melt(id_vars='Ticker', value_vars=['Retorno Anualizado (%)', 'Volatilidad Anualizada (%)'],
                          var_name='Métrica', value_name='Valor (%)')
fig_activos = px.bar(
    df_long,
    x='Ticker',
    y='Valor (%)',
    color='Métrica',
    barmode='group',
    title='Desempeño Individual de los Activos (Retorno vs Volatilidad)',
    color_discrete_sequence=['#2ecc71', '#e74c3c']
)
fig_activos.update_layout(template='plotly_dark')
fig_activos.show()

# 3. Matriz de Correlación
correlaciones = retornos.corr()
fig_heatmap = px.imshow(
    correlaciones,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="Matriz de Correlación de Activos (Retornos Diarios)"
)
fig_heatmap.update_layout(template='plotly_dark')
fig_heatmap.show()

## 6. Simulación de Evolución de Riqueza y Drawdown

Ejecutamos un backtesting unificado de riqueza histórica de las tres estrategias:
- Markowitz Rebalanceado Mensual con costos de transacción (0.10%).
- Equiponderado Rebalanceado Mensual con costos de transacción (0.10%).
- Buy & Hold con pesos de Markowitz sin costos extras ni rebalanceos.

In [8]:
frecuencia_sel = "Mensual"
codigo_freq = "ME"
factor_anual = 12
LAMBDA_TC = 0.0010

precios_res = precios.resample(codigo_freq).last()
retornos_res = precios_res.pct_change().dropna()

# Motor unificado de backtesting (idéntico al de app.py / M4)
def backtest_estrategia(retornos_df, w_objetivo, capital_inicial, modo="buy_hold", lambda_tc=0.0):
    R = retornos_df.to_numpy(dtype=float)
    T_len = R.shape[0]
    w_obj = np.asarray(w_objetivo, dtype=float)
    riqueza = np.zeros(T_len + 1)
    riqueza[0] = capital_inicial
    w_actual = w_obj.copy()
    
    for t in range(T_len):
        ret = R[t]
        if modo == "rebalancear" and t > 0:
            delta = np.sum(np.abs(w_actual - w_obj))
            tc = lambda_tc * delta
            cap_post = riqueza[t] * (1 - tc)
            ret_port = float(np.dot(w_obj, ret))
            riqueza[t + 1] = cap_post * (1 + ret_port)
            w_actual = w_obj * (1 + ret) / (1 + ret_port)
        else: # buy_hold
            ret_port = float(np.dot(w_actual, ret))
            riqueza[t + 1] = riqueza[t] * (1 + ret_port)
            w_actual = w_actual * (1 + ret) / (1 + ret_port)
    return riqueza

riqueza_eq = backtest_estrategia(retornos_res, np.ones(N)/N, CAPITAL_INICIAL, "rebalancear", LAMBDA_TC)
riqueza_mk = backtest_estrategia(retornos_res, w_sharpe, CAPITAL_INICIAL, "rebalancear", LAMBDA_TC)
riqueza_bh = backtest_estrategia(retornos_res, w_sharpe, CAPITAL_INICIAL, "buy_hold", 0.0)

fechas_sim = [precios_res.index[0]] + list(retornos_res.index)

# Gráfico de Riqueza
fig_sim = go.Figure()
fig_sim.add_trace(go.Scatter(x=fechas_sim, y=riqueza_mk, mode='lines', name=f'Markowitz Rebalanceado (${riqueza_mk[-1]:,.2f})', line=dict(color='red')))
fig_sim.add_trace(go.Scatter(x=fechas_sim, y=riqueza_eq, mode='lines', name=f'Equiponderado (${riqueza_eq[-1]:,.2f})', line=dict(color='blue')))
fig_sim.add_trace(go.Scatter(x=fechas_sim, y=riqueza_bh, mode='lines', name=f'Buy & Hold (${riqueza_bh[-1]:,.2f})', line=dict(color='gray')))
fig_sim.update_layout(
    title=f'Crecimiento de Riqueza por Estrategia (Rebalanceo {frecuencia_sel})',
    xaxis_title='Fecha',
    yaxis_title='Capital (USD)',
    template='plotly_dark'
)
fig_sim.show()

# Calcular drawdowns históricos
def calcular_drawdown(riqueza_path):
    cum_max = np.maximum.accumulate(riqueza_path)
    with np.errstate(divide='ignore', invalid='ignore'):
        dd = (riqueza_path - cum_max) / cum_max
        dd = np.nan_to_num(dd, nan=0.0)
    return dd * 100

dd_mk = calcular_drawdown(riqueza_mk)
dd_eq = calcular_drawdown(riqueza_eq)
dd_bh = calcular_drawdown(riqueza_bh)

# Gráfico de Drawdown
fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(x=fechas_sim, y=dd_mk, mode='lines', name='Drawdown Markowitz', line=dict(color='red')))
fig_dd.add_trace(go.Scatter(x=fechas_sim, y=dd_eq, mode='lines', name='Drawdown Equiponderado', line=dict(color='blue')))
fig_dd.add_trace(go.Scatter(x=fechas_sim, y=dd_bh, mode='lines', name='Drawdown Buy & Hold', line=dict(color='gray')))
fig_dd.update_layout(
    title=f'Caídas Máximas de las Estrategias (Drawdown - Rebalanceo {frecuencia_sel})',
    xaxis_title='Fecha',
    yaxis_title='Drawdown (%)',
    template='plotly_dark',
    hovermode="x unified"
)
fig_dd.show()

## 7. Exportación de Resultados

Exportamos los resultados detallados a un archivo JSON que será utilizado por otros módulos de rebalanceo y comparación, y guardamos los pesos óptimos del portafolio en un archivo Excel.

In [9]:
# Crear diccionario de métricas
metrics = {
    "tickers": TICKERS_VALIDOS,
    "configuracion": {
        "fecha_inicio": str(FECHA_INICIO),
        "fecha_fin": str(FECHA_FIN),
        "capital_inicial": CAPITAL_INICIAL,
        "frecuencia": frecuencia_sel,
        "dias_anio": DIAS_ANIO,
        "rf": RF
    },
    "markowitz_max_sharpe": {
        "retorno": float(r_s),
        "riesgo": float(v_s),
        "sharpe_teorico": float(sh_s),
        "pesos": dict(zip(TICKERS_VALIDOS, w_sharpe.round(6).tolist()))
    },
    "simulacion_riqueza_final": {
        "buy_and_hold": float(riqueza_bh[-1]),
        "equiponderado": float(riqueza_eq[-1]),
        "markowitz": float(riqueza_mk[-1])
    },
    "trayectorias": {
        "fechas": [d.strftime("%Y-%m-%d") for d in fechas_sim],
        "buy_and_hold": riqueza_bh.tolist(),
        "equiponderado": riqueza_eq.tolist(),
        "markowitz": riqueza_mk.tolist()
    }
}

# Guardar en raíz (para app.py)
with open("resultados_m1.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Guardar en salidas/ (para otros notebooks)
with open("salidas/resultados_m1.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Archivo resultados_m1.json exportado correctamente en ambos directorios.")

# Exportar a Excel
df_pesos_completo = pd.DataFrame({'Ticker': TICKERS_VALIDOS, 'Peso (%)': (w_sharpe * 100).round(2)})
excel_path = "salidas/pesos_markowitz.xlsx"
df_pesos_completo.to_excel(excel_path, index=False, sheet_name='Pesos_Optimos')
print(f"Pesos óptimos exportados correctamente a {excel_path}.")

Archivo resultados_m1.json exportado correctamente en ambos directorios.
Pesos óptimos exportados correctamente a salidas/pesos_markowitz.xlsx.
